# One-sided Hoeffding Inequality For Prior Likelihood Mixing

The Prior Likelihood Mixing Theorem is states that
$$
  C_t = \left\{\theta \in \Theta: L_t(\theta) \leq \frac{1}{\delta} - \log \int \prod_{s=1}^t p_s(y_s \mid \nu) \; d \mu_0(\nu)\right\}
$$
is a $(1-\delta)$-confidence sequence.

The one-sided version of Hoeffding's inequality (https://cs229.stanford.edu/extra-notes/hoeffding.pdf) applies to settings where we have independent random variables $Z_1, \dots, Z_n$ such that for all $i \in [n]$, there exist $a, b \in \mathbb{R}$ for which $a \leq Z_i \leq $. 
Hoeffding's theorem states that for all $t \in (0, \infty)$
$$
  \Pr\left(\frac{1}{n} \sum_{i=1}^n (Z_i - E[Z_i]) \geq t\right) \leq \exp\left(- \frac{2nt^2}{(b - a)^2}\right)
$$

Since we assume that for all time steps $s \in \mathbb{N}$, all images $\nu \in [0, 1]^{r^2}$ and all data sequences $((x_s, y_s), s \in \mathbb{N})$
$$
  p_s(y_s \mid \nu) = \prod_{i=1}^r \mathrm{Pois}(y_{s, i}, I_0 \exp(-\mathcal{R}(\nu, x_s)))
$$
we have that
$$
  \prod_{s=1}^t p_s(y_s \mid \nu) \in [0, 1].
$$

Define for all $i \in \mathbb{N}$, $Z_i \coloneqq \prod_{s=1}^t p_s(y_s \mid \vartheta)$ and let $\vartheta \sim \mu_0$. 
Furthermore, assume that all $Z_1, \dots, Z_n$ are i.i.d. Then applying Hoeffding's inequality yields
$$
  \Pr\left(\frac{1}{n} \sum (Z_i - E Z_i) \geq t\right) \leq \exp\left(- \frac{2nt^2}{(b_i - a_i)^2}\right)
$$
The i.i.d. assumption implies
$$
  \Pr\left(\frac{1}{n} \sum (Z_i - E Z_i) \geq t\right) = \Pr\left(-t + \frac{1}{n} \sum Z_i \geq E Z_1 \right) \leq \exp\left(- \frac{2nt^2}{(b - a)^2}\right).
$$
Let 
$$
  \delta = \exp\left(- \frac{2nt^2}{(b - a)^2}\right).
$$
Choose error tolerance $t = 0.01$, error level $\delta \leq 0.025$. 
Then plugging $a, b$ and $t$ into the RHS yields
$$
  \log \delta = -0.0002n \leq \log 0.025 \iff n \geq 18444.397\dots
$$
So a sample size of $n = 18445$ suffices.

## Sequential Application of Hoeffding's Inequality

For each round $r=1,\dots,200$, define

$$
Z_i^{(r)} \coloneqq \prod_{s=1}^{t_r} p_s(y_s\mid \vartheta_i)\in[0,1],\qquad
\hat Z^{(r)} \coloneqq \frac{1}{n}\sum_{i=1}^n Z_i^{(r)}.
$$

By one-sided Hoeffding (with $a=0,b=1$), for any fixed $r$ and any deviation level $t>0$,

$$
\Pr\left( \hat{Z}^{(r)}- \mathbb{E}[Z_1^{(r)}] \geq t \right) \leq \exp(-2nt^2).
$$

Apply a union bound over the 200 rounds:

$$
\Pr\left( \exists r \in [200] : \hat{Z}^{(r)}- \mathbb{E}[Z_1^{(r)}] \geq t \right)
= \Pr\left( \exists r \in [200] : -t + \hat{Z}^{(r)} \geq \mathbb{E}[Z_1^{(r)}] \right)
\leq 200 \exp(-2nt^2).
$$

To ensure this probability is at most $\delta = 0.025$, it suffices that

$$
200 \exp(-2nt^2) \leq \delta
\iff
n \geq \frac{1}{2t^{2}}\ln \left(\frac{200}{\delta}\right).
$$

With your choices $t=0.01$ and $\delta=0.025$,

$$
n \ge \frac{1}{2(0.01)^2} \ln \left(\frac{200}{0.025}\right)
= 5000\ln(8000) \approx 44935.98.
$$

Therefore, $n=44936$ samples suffice to guarantee, with probability at least $97.5\%$, that for all $r=1,\dots,200$,

$$
\mathbb{E}[Z_1^{(r)}] \geq \hat Z^{(r)} - 0.01.
$$

Then
$$
  -\log \mathbb{E}[Z_1^{(r)}] \leq -\log(\hat Z^{(r)} - 0.01).
$$
with with probability at least $97.5\%$ as well (if $\hat Z^{(r)} - 0.01 > 0$ for all $r$).



In [31]:
# Parameters from the write-up (corrected & executable)
import math
import pandas as pd

# Single-round parameters
a, b = 0.0, 1.0
t = 0.01           # deviation level (epsilon)
delta_single = 0.025   # error level for single round

# Multi-round parameters
R = 200                # number of rounds
delta_multi = 0.025    # overall error level across all rounds

def hoeffding_one_sided_n(a, b, eps, delta):
    """
    Minimum n for one-sided Hoeffding bound (right tail):
    P( (1/n) * sum_i (Z_i - E[Z_i]) >= eps ) <= delta,
    for Z in [a,b].
    
    n >= ((b-a)^2 / (2 * eps^2)) * ln(1/delta)
    """
    width = b - a
    return (width**2) / (2 * eps**2) * math.log(1/delta)

def hoeffding_union_over_rounds_n(a, b, eps, delta, rounds):
    """
    Minimum n so the one-sided Hoeffding bound holds simultaneously
    over `rounds` rounds via a union bound:
    
    P( exists r in [rounds] : (1/n) * sum_i (Z_i^{(r)} - E[Z_i^{(r)}]) >= eps ) <= delta
    
    n >= ((b-a)^2 / (2 * eps**2)) * ln(rounds / delta)
    """
    width = b - a
    return (width**2) / (2 * eps**2) * math.log(rounds / delta)

# Compute results
n_single = hoeffding_one_sided_n(a, b, t, delta_single)
n_multi  = hoeffding_union_over_rounds_n(a, b, t, delta_multi, R)

# Collect detailed intermediate numbers
results = pd.DataFrame([
    {
        "Scenario": "Single round",
        "a": a, "b": b, "t": t, "delta": delta_single, "rounds": 1,
        "ln(1/delta)": math.log(1/delta_single),
        "1/(2*t^2)": 1/(2*t**2),
        "n (exact)": n_single,
        "n (ceil)": math.ceil(n_single),
    },
    {
        "Scenario": f"Uniform over {R} rounds",
        "a": a, "b": b, "t": t, "delta": delta_multi, "rounds": R,
        "ln(rounds/delta)": math.log(R/delta_multi),
        "1/(2*t^2)": 1/(2*t**2),
        "n (exact)": n_multi,
        "n (ceil)": math.ceil(n_multi),
    },
])


print("=== Single round ===")
print(f"t = {t}, delta = {delta_single}")
print(f"n >= ((b-a)^2 / (2 t^2)) * ln(1/delta)")
print(f"n >= {1/(2*t**2):.0f} * ln(1/{delta_single}) = {n_single:.2f}")
print(f"n_ceiling = {math.ceil(n_single)}")

print("\n=== Uniform over R rounds via union bound ===")
print(f"R = {R}, t = {t}, delta = {delta_multi}")
print(f"n >= ((b-a)^2 / (2 t^2)) * ln(R/delta)")
print(f"n >= {1/(2*t**2):.0f} * ln({R}/{delta_multi}) = {n_multi:.2f}")
print(f"n_ceiling = {math.ceil(n_multi)}")


=== Single round ===
t = 0.01, delta = 0.025
n >= ((b-a)^2 / (2 t^2)) * ln(1/delta)
n >= 5000 * ln(1/0.025) = 18444.40
n_ceiling = 18445

=== Uniform over R rounds via union bound ===
R = 200, t = 0.01, delta = 0.025
n >= ((b-a)^2 / (2 t^2)) * ln(R/delta)
n >= 5000 * ln(200/0.025) = 44935.98
n_ceiling = 44936


In [33]:
# Simulation: guarantee P(hatZ - t <= E[Z]) (i.e., hatZ - E[Z] <= t) with high probability.
# We check violations of the *right-tail* event: hatZ - E[Z] >= t.

import math
import numpy as np
import pandas as pd

def hoeffding_union_over_rounds_n(a, b, eps, delta, rounds):
    width = b - a
    return (width**2) / (2 * eps**2) * math.log(rounds / delta)

# Parameters
R = 200
delta = 0.025
a, b = 0.0, 1.0
t = 0.01
n = math.ceil(hoeffding_union_over_rounds_n(a, b, t, delta, R))

# Generator: Beta(2,5)
rng = np.random.default_rng(20250922)
alpha, beta_param = 2.0, 5.0
EZ_true = alpha / (alpha + beta_param)

# Simulate
hatZ = np.array([rng.beta(alpha, beta_param, size=n).mean() for _ in range(R)])

# Right-tail gaps: hatZ - E[Z]
gap_right = hatZ - EZ_true

# Violations of desired guarantee hatZ - t <= E[Z]  <=>  hatZ - E[Z] <= t
violations = int(np.sum(gap_right >= t))

df = pd.DataFrame({
    "round": np.arange(1, R+1),
    "hatZ": hatZ,
    "diagnostic_EZ_true": EZ_true,
    "gap_right_hatZ_minus_EZ": gap_right,
    "uniform_eps_t": t,
    "violation_(gap_right>=t)": gap_right >= t,
})

print("=== Parameters ===")
print(f"R={R}, n={n}, delta={delta}, [a,b]=[{a},{b}]")
print(f"t = {t}")
print(f"Required n by union bound: {n}")

print("\n=== Diagnostics (right tail) ===")
print(f"True E[Z] = {EZ_true:.6f}")
print(f"Max (hatZ - E[Z]) over rounds = {gap_right.max():.6f}")
print(f"#violations of (hatZ - E[Z] >= t) across {R} rounds = {violations}")


=== Parameters ===
R=200, n=44936, delta=0.025, [a,b]=[0.0,1.0]
t = 0.01
Required n by union bound: 44936

=== Diagnostics (right tail) ===
True E[Z] = 0.285714
Max (hatZ - E[Z]) over rounds = 0.002352
#violations of (hatZ - E[Z] >= t) across 200 rounds = 0
